# `PT_results.ipynb`

This notebook is for plotting cached `C_\ell^{\phi T}` results. It should not be used to recompute the expensive simulation averages interactively.

## Run Order

1. Run the main scenario jobs with Slurm so the `PLENS` products exist.
2. Run the cache-building Slurm job so the notebook has compact `.npz` files to load.
3. Open this notebook with the `delens-env` Jupyter kernel.
4. Run the cells from top to bottom.

## Required Slurm Jobs

The notebook assumes the scenario outputs already exist under `$PLENS`. The thesis-aligned scenario production jobs were submitted via `run_single_scenario.slurm`.

To build the compact plot cache used below, run:

```bash
cd /home3/p283342/Delensing/clean-delensing
sbatch compute_pt_plot_data.slurm
```

If you only want one group:

```bash
sbatch --export=ALL,GROUP=noiseless compute_pt_plot_data.slurm
sbatch --export=ALL,GROUP=noisy compute_pt_plot_data.slurm
```

## What You Must Change On Another Machine Or Cluster

These paths are cluster-specific and will need to be edited if you run elsewhere:

- `repo_root`: location of this repository
- `PLENS`: directory containing the cached Planck lensing products
- `INPUT`: directory containing the downloaded CMB, noise, and SMICA inputs
- `PARAMS`: directory containing masks and `dcl_*` files
- `KFIELD`: directory containing the input lensing maps
- `cache_dir`: directory containing the `.npz` files written by `compute_pt_plot_data.py`

If those paths are wrong, the notebook will either fail to import the project modules or fail to find the cached plot data.


## Kernel And Path Setup

This cell does three things:

- adds the repository root to `sys.path` so `env_config.py` and the project modules are importable
- sets the environment variables that the project expects
- prints the active Python executable so you can confirm that the notebook is running in `delens-env`

On another machine, this is the first place you will need to edit paths.


In [1]:
import os
import sys
from pathlib import Path

# Edit this if the repository lives somewhere else on your system.
repo_root = Path("/home3/p283342/Delensing/clean-delensing")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Edit these paths if you move the data products or run on a different cluster.
os.environ["PLENS"] = "/scratch/hb-CosmoGroup/Delensing/PLENS"
os.environ["INPUT"] = "/scratch/hb-CosmoGroup/Delensing/INPUT"
os.environ["PARAMS"] = "/home3/p283342/Delensing/clean-delensing/input"
os.environ["KFIELD"] = "/scratch/hb-CosmoGroup/Delensing/KFIELD"

print("Python executable:", sys.executable)
print("PLENS:", os.environ.get("PLENS"))
print("INPUT:", os.environ.get("INPUT"))
print("PARAMS:", os.environ.get("PARAMS"))
print("KFIELD:", os.environ.get("KFIELD"))


Python executable: /home3/p283342/delens-env/bin/python
PLENS: /scratch/hb-CosmoGroup/Delensing/PLENS
INPUT: /scratch/hb-CosmoGroup/Delensing/INPUT
PARAMS: /home3/p283342/Delensing/clean-delensing/input
KFIELD: /scratch/hb-CosmoGroup/Delensing/KFIELD


## Imports And Cache Loading

This cell imports the plotting helpers and loads the cached `.npz` files created by `compute_pt_plot_data.py`.

The important path here is `cache_dir`. If you changed the output directory in `compute_pt_plot_data.py` or in the Slurm wrapper, update it here as well.

The loaded objects are reused by all plot cells below, so the notebook stays fast after the expensive aggregation has been done once in Slurm.


In [2]:
import numpy as np
import matplotlib.pyplot as plt

import env_config
from binner_sims import binner_sims
import thesis_plot_style as tps
import parfiles.noNoise.parfile as par_nn
import parfiles.Noise.parfile as par_n

bin_type = "fullA_10"
tps.apply_style("single")
cmap = tps.CB_PALETTE

# Edit this if you wrote the cached phi-T plot inputs somewhere else.
cache_dir = Path("/home3/p283342/Delensing/clean-delensing/THESIS/cache/pt_results")

noiseless_fid = np.load(cache_dir / "noiseless_fiducial.npz", allow_pickle=True)
noisy_fid = np.load(cache_dir / "noisy_fiducial.npz", allow_pickle=True)

noiseless_datasets = [
    np.load(cache_dir / "noiseless_mv_lensed.npz", allow_pickle=True),
    np.load(cache_dir / "noiseless_mv_input_kappa.npz", allow_pickle=True),
    np.load(cache_dir / "noiseless_mv_internal_qest.npz", allow_pickle=True),
    np.load(cache_dir / "noiseless_tt_internal_polqest.npz", allow_pickle=True),
]

noisy_datasets = [
    np.load(cache_dir / "noisy_mv_lensed.npz", allow_pickle=True),
    np.load(cache_dir / "noisy_mv_input_kappa.npz", allow_pickle=True),
    np.load(cache_dir / "noisy_mv_internal_qest.npz", allow_pickle=True),
    np.load(cache_dir / "noisy_tt_internal_polqest.npz", allow_pickle=True),
]


Using lenspyx alm2map


FileNotFoundError: [Errno 2] No such file or directory: '/home3/p283342/Delensing/clean-delensing/THESIS/cache/pt_results/noisy_fiducial.npz'

## Figure: Noiseless `C_\ell^{\phi T}`

This cell plots the noiseless thesis comparison figure:

- fiducial theory curve
- lensed baseline
- delensed with input `\kappa_{LM}`
- internally delensed with MV-QEST
- internally delensed with Pol-QEST

All four Monte Carlo curves are loaded from the cache rather than recomputed here.


In [ ]:
with tps.context_figure(filename="fig_clpt_noiseless.png") as (fig, ax):
    ax.plot(
        noiseless_fid["ell_full"],
        noiseless_fid["cl_fid"],
        "k--",
        lw=1.6,
        label="fiducial",
    )

    for ci, d in enumerate(noiseless_datasets):
        ax.errorbar(
            d["ell"],
            d["cl_mean"],
            yerr=d["cl_err"],
            fmt="o-",
            lw=1.6,
            capsize=3,
            color=tps.CB_PALETTE[ci + 2],
            label=str(d["label"]),
        )

    ax.set(
        xlabel=r"$\ell$",
        ylabel=r"$10^2\,\ell^3\,C_\ell^{\phi T}\ [\mu{m K}]$",
        title="Noiseless $C_\ell^{\phi T}$ Estimates",
    )
    ax.set_xlim([0, 105])
    ax.legend(loc="upper right")


## Figure: Noisy `C_\ell^{\phi T}`

This is the noisy counterpart of the previous figure. The same four scenario families are compared, but now with the noisy parameter files.

Again, the notebook only reads cached `.npz` products.


In [ ]:
with tps.context_figure(filename="fig_clpt_noisy.png") as (fig, ax):
    ax.plot(
        noisy_fid["ell_full"],
        noisy_fid["cl_fid"],
        "k--",
        lw=1.6,
        label="fiducial",
    )

    for ci, d in enumerate(noisy_datasets):
        ax.errorbar(
            d["ell"],
            d["cl_mean"],
            yerr=d["cl_err"],
            fmt="o-",
            lw=1.6,
            capsize=3,
            color=tps.CB_PALETTE[ci + 2],
            label=str(d["label"]),
        )

    ax.set(
        xlabel=r"$\ell$",
        ylabel=r"$10^2\,\ell^3\,C_\ell^{\phi T}\ [\mu{m K}]$",
        title="Noisy $C_\ell^{\phi T}$ Estimates",
    )
    ax.set_xlim([0, 105])
    ax.legend(loc="upper right")


## Figure: Noiseless Wiener Filters And `\phi`-`T` Efficiencies

This cell mixes two ingredients:

- live Wiener filter curves from the `binner_sims` PP helper
- cached `\phi`-`T` spectra from the `.npz` files

The efficiency curves are derived from the cached baseline and delensed spectra, so the notebook does not need to loop over simulations here.


In [ ]:
d0 = noiseless_datasets[0]
d1 = noiseless_datasets[2]
d2 = noiseless_datasets[3]

with tps.context_figure(filename="fig_wf2_eff_phiT_noiseless.png") as (fig, ax):
    ax.axvspan(25, 35, color="gray", alpha=0.2, zorder=0)

    bw_pol = binner_sims("p_p", "p_p", par_nn, bin_type)
    bw_mv = binner_sims("p", "p", par_nn, bin_type)

    ax.plot(np.arange(2049), bw_mv.get_WF() ** 0.5,
            color=cmap[2], lw=1.6,
            label="Wiener $W_\ell$ (MV-QE)")
    ax.plot(np.arange(2049), bw_pol.get_WF() ** 0.5,
            color=cmap[4], lw=1.6,
            label="Wiener $W_\ell$ (Pol-QE)")

    C0, s0 = d0["cl_mean"], d0["cl_err"]
    C1, s1 = d1["cl_mean"], d1["cl_err"]
    C2, s2 = d2["cl_mean"], d2["cl_err"]
    ell = d0["ell"]

    eff1 = 1.0 - C1 / C0
    err1 = np.sqrt((s1 / C0) ** 2 + (C1 * s0 / C0 ** 2) ** 2)

    eff2 = 1.0 - C2 / C0
    err2 = np.sqrt((s2 / C0) ** 2 + (C2 * s0 / C0 ** 2) ** 2)

    ax.errorbar(ell, eff1, yerr=err1,
                fmt="o-", capsize=3, lw=1.4,
                color=cmap[3],
                label=r"$\epsilon_{m MV\!	o\!MV	ext{-}QEST}$")
    ax.errorbar(ell, eff2, yerr=err2,
                fmt="o-", capsize=3, lw=1.4,
                color=cmap[5],
                label=r"$\epsilon_{m TT\!	o\!Pol	ext{-}QEST}$")

    ax.hlines(0, 0, 300, "grey", "--", alpha=0.3)
    ax.set(xlabel=r"$\ell$",
           ylabel="Dimensionless",
           title="Wiener Filters & φ–T Efficiencies (Noiseless)")
    ax.set_xlim(0, 105)
    ax.set_ylim(-0.1, 1)
    ax.legend(loc="lower right", frameon=False)


## Figure: Noisy Wiener Filters And `\phi`-`T` Efficiencies

This is the noisy analogue of the previous figure.

If this cell fails on another system, first check:

- whether `cache_dir` points to the correct `pt_results` cache
- whether the noisy scenario jobs finished successfully under `$PLENS`
- whether the notebook kernel is still using the intended project environment


In [ ]:
d0 = noisy_datasets[0]
d1 = noisy_datasets[2]
d2 = noisy_datasets[3]

with tps.context_figure(filename="fig_wf2_eff_phiT_noisy.png") as (fig, ax):
    ax.axvspan(25, 35, color="gray", alpha=0.2, zorder=0)

    bw_pol = binner_sims("p_p", "p_p", par_n, bin_type)
    bw_mv = binner_sims("p", "p", par_n, bin_type)
    ell_W = np.arange(4097)

    ax.plot(ell_W, bw_mv.get_WF() ** 0.5,
            color=cmap[2], lw=1.6,
            label="Wiener $W_\ell$ (MV-QE)")
    ax.plot(ell_W, bw_pol.get_WF() ** 0.5,
            color=cmap[4], lw=1.6,
            label="Wiener $W_\ell$ (Pol-QE)")

    C0, s0 = d0["cl_mean"], d0["cl_err"]
    C1, s1 = d1["cl_mean"], d1["cl_err"]
    C2, s2 = d2["cl_mean"], d2["cl_err"]
    ell = d0["ell"]

    eff1 = 1.0 - C1 / C0
    err1 = np.sqrt((s1 / C0) ** 2 + (C1 * s0 / C0 ** 2) ** 2)

    eff2 = 1.0 - C2 / C0
    err2 = np.sqrt((s2 / C0) ** 2 + (C2 * s0 / C0 ** 2) ** 2)

    ax.errorbar(ell, eff1, yerr=err1,
                fmt="o-", capsize=3, lw=1.4,
                color=cmap[3],
                label=r"$\epsilon_{m MV\!	o\!MV	ext{-}QEST}$")
    ax.errorbar(ell, eff2, yerr=err2,
                fmt="o-", capsize=3, lw=1.4,
                color=cmap[5],
                label=r"$\epsilon_{m TT\!	o\!Pol	ext{-}QEST}$")

    ax.hlines(0, 0, 300, "grey", "--", alpha=0.3)
    ax.set(xlabel=r"$\ell$",
           ylabel="Dimensionless",
           title="Wiener Filters & φ–T Efficiencies (Noisy)")
    ax.set_xlim(0, 105)
    ax.set_ylim(-0.1, 1)
    ax.legend(loc="upper right", frameon=False)
